# SmolLM2-recurrent continued pretraining

Upload this notebook to Colab (Runtime > Change runtime type > GPU) and Run All. No editing required -- except the one `MODE` line in the cell below.

One-time setup, before the first run: open the key icon in the left sidebar (Secrets) and add:
- `HF_TOKEN` (required) -- a Hugging Face token with **write** access, from https://huggingface.co/settings/tokens. Used to pull/push checkpoints and the final model, so no Drive mount is needed.
- `WANDB_API_KEY` (optional) -- from https://wandb.ai/authorize. If missing, training still runs fine, just prints loss to the cell output instead of logging to wandb.

`MODE = "smoke_test"` (the default) runs ~200 steps on a couple hundred MB of data -- just enough to prove the pipeline works end-to-end (data mix, train, checkpoint push, resume, eval) and to sanity-check the loss and recurrence-depth trend before spending real GPU hours. Flip it to `MODE = "full_run"` for the actual ~7500-step / ~500M-token continued-pretraining run once the smoke test looks good.

Everything is on the Hub, not Google Drive, and everything is namespaced by run name so the two modes never collide or resume from each other's state:
- Training data re-mixes into local (ephemeral) Colab disk each session -- it isn't persisted, so a fresh session re-downloads/re-mixes it.
- The **latest** full resumable checkpoint (weights + optimizer + dataloader state) for the current run is pushed to the private dataset repo `usr-wwelsh/smollm2-recurrent-checkpoints` every save, overwriting that run's previous one -- so it never accumulates, and there's always exactly one copy to resume from.
- Every time you reopen this notebook and Run All again (same `MODE`), it checks that repo for a checkpoint under the current run name and resumes from it instead of starting over. Expect to need this a lot -- free-tier Colab sessions time out well before a full run finishes, and `smoke_test` checkpoints deliberately save often (every 25 steps) so you can kill the runtime mid-test and confirm resume actually works.
- Only `full_run` pushes the finished weights to the public model repo `usr-wwelsh/Recurrent-SmolLM2-360M-4-14-4-trained` -- `smoke_test` never publishes anything.

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU visible. Go to Runtime > Change runtime type and select a GPU, "
    "then Runtime > Restart session and run this cell again."
)

gpu_name = torch.cuda.get_device_name(0)
major, _minor = torch.cuda.get_device_capability(0)
no_amp = "false" if major >= 8 else "true"
print(f"GPU: {gpu_name} (compute capability {major}.{_minor}) -> no_amp={no_amp}")
print(
    "bf16 autocast enabled." if no_amp == "false" else
    "Pre-Ampere GPU (T4/P100) -- no native bf16, running in fp32 for correctness (slower per-step, but correct)."
)

In [ ]:
# ============================================================================
# The one line to change: "smoke_test" to prove the pipeline works cheaply,
# "full_run" for the real continued-pretraining run. Everything below derives
# from this -- run name, step count, token budget, checkpoint frequency.
MODE = "smoke_test"
# ============================================================================

if MODE == "smoke_test":
    RUN_NAME = "smollm2-recurrent-smoketest"
    MAX_STEPS = 200
    TOKEN_BUDGET = 15_000_000  # 200 steps * batch_size=64 * max_length=1024 needs ~13.1M tokens minimum
    SAVE_INTERVAL = 25  # eval checkpoint every 25 steps, full resumable checkpoint every 50 -- frequent on purpose, so a session kill mid-test actually exercises resume
    ROWS_PER_SHARD = 2_000  # ~14.6k rows expected total -- flush every ~2M tokens instead of only at the very end, so a kill mid-mix still leaves usable shards and you see "wrote shard-*" early instead of nothing
    LOG_EVERY = 200  # default (1000) can go a couple minutes with zero output on a small budget -- looks hung even though it isn't
    SHUFFLE_BUFFER_SIZE = 0  # streaming .shuffle() yields nothing until its buffer fills -- default 10,000 means 3 sources * 10k = 30k documents downloaded before the FIRST packed row, dwarfing this test's ~14.6k total rows. 0 disables shuffling (mix ordering is a non-issue at this scale) so packing starts immediately
    DCLM_WEIGHT = 0  # HuggingFaceTB/dclm-edu ships ~2.9GB shards as a single Parquet row group each -- reading even one row needs ~9GB+ RAM, which OOM-kills a Colab instance for no benefit on a 200-step pipeline check. Skipping it here just reweights the mix to fineweb-edu/cosmopedia-v2; only full_run needs the real 60/40/4 ratio
elif MODE == "full_run":
    RUN_NAME = "smollm2-recurrent-v1"
    MAX_STEPS = 7500
    TOKEN_BUDGET = 500_000_000
    SAVE_INTERVAL = 250
    ROWS_PER_SHARD = 20_000  # default -- unchanged from the original design
    LOG_EVERY = 1_000
    SHUFFLE_BUFFER_SIZE = 10_000  # default -- fine here, the 30k-doc prefetch is negligible against 500M tokens
    DCLM_WEIGHT = 0.40  # default -- full_run needs the real 60/40/4 mix. mix_smollm2_corpus.py disables the Hub's Xet transfer backend itself to bound memory to one ~2.9GB shard at a time instead of many concurrent ones, but this still peaks around 9-10GB RAM per DCLM shard -- use a Colab High-RAM/Pro runtime for full_run
else:
    raise ValueError(f"Unknown MODE={MODE!r}, expected 'smoke_test' or 'full_run'")

print(f"MODE={MODE}: run_name={RUN_NAME}, max_steps={MAX_STEPS:,}, token_budget={TOKEN_BUDGET:,}, save_interval={SAVE_INTERVAL}, shuffle_buffer_size={SHUFFLE_BUFFER_SIZE}, dclm_weight={DCLM_WEIGHT}")

In [ ]:
import os
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get("HF_TOKEN"))

# Everything below is local Colab disk (ephemeral -- wiped when the session ends).
# Persistence across sessions comes entirely from the Hub repos, not this filesystem.
DATA_PATH = f"/content/data/{RUN_NAME}"
OUT_PATH = "/content/huginn_smollm2"

CHECKPOINT_REPO = "usr-wwelsh/smollm2-recurrent-checkpoints"   # private HF dataset repo, one file per run_name -- holds only the latest resumable checkpoint for each
FINAL_MODEL_REPO = "usr-wwelsh/Recurrent-SmolLM2-360M-4-14-4-trained"   # public HF model repo -- only written to in "full_run" mode

os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(OUT_PATH, exist_ok=True)
print(f"Local data dir: {DATA_PATH}")
print(f"Local checkpoint dir: {OUT_PATH}/{RUN_NAME}")
print(f"Resume checkpoints from/to: {CHECKPOINT_REPO}/{RUN_NAME}")

In [ ]:
import os

REPO_DIR = "/content/retrofitting-recurrence"
if not os.path.exists(REPO_DIR):
    !git clone --depth=1 https://github.com/usr-wwelsh/retrofitting-recurrence.git {REPO_DIR}
    assert _exit_code == 0, f"git clone failed with exit code {_exit_code} -- check the output above"
%cd {REPO_DIR}
!git pull
assert _exit_code == 0, f"git pull failed with exit code {_exit_code} -- check the output above"
!pip install -q -r requirements.txt
assert _exit_code == 0, f"pip install failed with exit code {_exit_code} -- check the output above for which package broke"


In [ ]:
wandb_disabled = "true"
try:
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    wandb_disabled = "false"
    print("WANDB_API_KEY secret found -- wandb logging enabled.")
except Exception as e:
    print(f"No usable WANDB_API_KEY secret ({e}) -- wandb logging disabled, training still runs fine.")

In [ ]:
import glob

if glob.glob(f"{DATA_PATH}/*.parquet"):
    print(f"Found existing packed shards in {DATA_PATH}, skipping the mix step.")
else:
    mix_desc = "FineWeb-Edu/Cosmopedia-v2" if DCLM_WEIGHT == 0 else "FineWeb-Edu/DCLM/Cosmopedia-v2"
    print(f"No packed data found -- streaming and packing ~{TOKEN_BUDGET:,} tokens of the {mix_desc} mix.")
    print("The first 'Resolving data files' lines are just the Hub listing each source's parquet shards (up to ~1769 for DCLM, when included) -- normal, not a hang.")
    print("Then it goes quiet again until it's actually packed a full shard or hit log_every docs -- give it a minute rather than interrupting; -u below makes that output show up as it happens instead of buffering.")
    print("Resumes cleanly if interrupted anyway (already-written shards are kept, just re-run this cell) -- interrupting only loses whatever wasn't flushed to a shard yet.")
    !python -u mix_smollm2_corpus.py --save_path="{DATA_PATH}" --token_budget={TOKEN_BUDGET} --rows_per_shard={ROWS_PER_SHARD} --log_every={LOG_EVERY} --shuffle_buffer_size={SHUFFLE_BUFFER_SIZE} --dclm_weight={DCLM_WEIGHT}
    assert _exit_code == 0, f"mix_smollm2_corpus.py failed with exit code {_exit_code} -- scroll up for the traceback, then re-run this cell (already-written shards are kept)."

# belt-and-suspenders: don't let a silently-empty mix step reach training
assert glob.glob(f"{DATA_PATH}/*.parquet"), f"No .parquet shards found in {DATA_PATH} after the mix step -- check the output above for what went wrong."


In [ ]:
from huggingface_hub import hf_hub_download, HfApi

resume_flag = ""
api = HfApi()
if api.repo_exists(CHECKPOINT_REPO, repo_type="dataset"):
    try:
        resume_dir = f"{OUT_PATH}/resumed_checkpoint"
        hf_hub_download(repo_id=CHECKPOINT_REPO, repo_type="dataset", filename=f"{RUN_NAME}/chkpt.pt", local_dir=resume_dir)
        resume_flag = f"--resume_path={resume_dir}/{RUN_NAME}"
        print(f"Found a checkpoint for {RUN_NAME} on {CHECKPOINT_REPO} -- resuming from it.")
    except Exception as e:
        print(f"No checkpoint for {RUN_NAME} on {CHECKPOINT_REPO} yet ({e}) -- starting a fresh run.")
else:
    print(f"No checkpoint on {CHECKPOINT_REPO} yet -- starting a fresh run.")

In [ ]:
!python -u train.py \
    --run_name={RUN_NAME} \
    --out_path={OUT_PATH} \
    --model_name=usr-wwelsh/Recurrent-SmolLM2-360M-4-14-4 \
    --hub_checkpoint_repo={CHECKPOINT_REPO} \
    --preprocessed_data_path={DATA_PATH} \
    --is_parquet_dataset=true \
    --max_length=1024 \
    --micro_batch_size=8 \
    --batch_size=64 \
    --optim_config.lr=5e-5 \
    --scheduler_args.warmup=0.02 \
    --scheduler_args.cooldown=0.9 \
    --max_grad_norm=1.0 \
    --no_amp={no_amp} \
    --max_steps={MAX_STEPS} \
    --compile=false \
    --save_interval={SAVE_INTERVAL} \
    --wandb_disabled={wandb_disabled} \
    --mean_recurrence_schedule.turn_on=true \
    --mean_recurrence_schedule.warmup=0.25 \
    --mean_recurrence_schedule.max_mean_rec=4 \
    {resume_flag}
assert _exit_code == 0, f"train.py exited with code {_exit_code} -- check the traceback above. Cells below did not see a finished run."


## Push final model to the Hub

Only runs in `full_run` mode, and only once training has actually reached `MAX_STEPS` in some session (checked via the checkpoint's saved step count, not just file presence) -- if the training cell above got cut off by a session timeout, re-run this notebook (Run All) to resume training first.

In [ ]:
import re

if MODE != "full_run":
    print(f"MODE={MODE!r} -- skipping the Hub push (only full_run publishes to {FINAL_MODEL_REPO}).")
else:
    ckpt_dirs = glob.glob(f"{OUT_PATH}/{RUN_NAME}/model_only_chkpt_*")
    if not ckpt_dirs:
        print("No eval checkpoint found yet -- nothing to push.")
    else:
        last_step = max(int(re.search(r"model_only_chkpt_(\d+)", d).group(1)) for d in ckpt_dirs)
        if last_step < MAX_STEPS:
            print(f"Latest checkpoint is at step {last_step:,}/{MAX_STEPS:,} -- training isn't finished yet, re-run this notebook to resume before pushing.")
        else:
            ckpt_path = f"{OUT_PATH}/{RUN_NAME}/model_only_chkpt_{last_step}"
            print(f"Training complete at step {last_step:,} -- pushing {ckpt_path} to {FINAL_MODEL_REPO}")
            api.create_repo(FINAL_MODEL_REPO, private=False, exist_ok=True)
            api.upload_folder(repo_id=FINAL_MODEL_REPO, folder_path=ckpt_path, commit_message=f"trained checkpoint @ step {last_step}")
            print("Done.")

## Eval

Sweeps recurrence depth on the latest local eval checkpoint and compares against base SmolLM2-360M, so you can see whether training is recovering toward (not stuck below) the base model. Works on any checkpoint reached so far -- doesn't require training to have finished, and works in either `MODE` (though `smoke_test` numbers are only a pipeline sanity check, not a real signal).

In [ ]:
ckpt_dirs = glob.glob(f"{OUT_PATH}/{RUN_NAME}/model_only_chkpt_*")
if not ckpt_dirs:
    print("No model_only checkpoint found yet (training hasn't reached a save_interval boundary) -- nothing to eval.")
else:
    last_step = max(int(re.search(r"model_only_chkpt_(\d+)", d).group(1)) for d in ckpt_dirs)
    ckpt_path = f"{OUT_PATH}/{RUN_NAME}/model_only_chkpt_{last_step}"
    print(f"Evaluating {ckpt_path}")

    TASKS = "arc_easy,arc_challenge,hellaswag,mmlu,piqa,winogrande"
    for mean_recurrence in [1, 2, 4, 8]:
        out_dir = f"eval_outputs/{RUN_NAME}/step_{last_step}/mean_recurrence_{mean_recurrence}"
        !lm_eval --model hf \
            --model_args pretrained={ckpt_path},mean_recurrence={mean_recurrence},add_bos_token=True,dtype="float32",trust_remote_code=True \
            --tasks {TASKS} \
            --device cuda \
            --output_path {out_dir} \
            --batch_size auto

    !lm_eval --model hf \
        --model_args pretrained=HuggingFaceTB/SmolLM2-360M,add_bos_token=True,dtype="float32" \
        --tasks {TASKS} \
        --device cuda \
        --output_path eval_outputs/SmolLM2-360M-base \
        --batch_size auto